# 01 遅延評価と実行計画

**目的**: DataFrame API は「処理を実行する API」ではなく「実行計画を組み立てる API」だと、手を動かして確かめる（60分）。

終わったときに説明できるようになること:

- transformation と action の違い。計算はいつ走るのか
- `explain()` の読み方。書いたコードと、実際に実行される計画がどう違うか
- 列の刈り込み、述語プッシュダウン、パーティションの刈り込み
- 最適化が効かなくなる書き方（UDF）と、`cache()` の効きどころ

**進め方**: 各実験で、先に予想を書く → 実行する → 観察を書く。`# TODO` のセルは `...` を自分で埋める。直後のセルが ✅ / ❌ で確かめてくれる。

In [ ]:
import shutil

from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType

from labkit import DATA, check_plan, get_spark, large, materialize, observe, same_optimized_plan, timed

spark = get_spark("01_lazy_evaluation", {"spark.sql.adaptive.enabled": "false"})

orders = large(spark, "orders")
customers = large(spark, "customers")

`spark.sql.adaptive.enabled`（AQE）を切っているのは、Spark が実行中に計画を書き換える機能を止めて、素の挙動を見るため。AQE は `03_shuffle` の最後で ON にする。

---

## 実験1: transformation では何も実行されない

✏️ **予想: 500万行の注文を filter して groupBy する DataFrame を「定義する」のに何秒かかる？ そのあと count() を呼ぶと？**

1秒くらい？countすると10秒くらい？

In [ ]:
# TODO 1-1: completed な注文だけを残し、顧客ごとの注文数を数える DataFrame を作る
#   使うもの: orders.filter(...) / .groupBy("customer_id") / .agg(F.count("*").alias("n_orders"))
#   ここではまだ show() や count() を呼ばない
with timed("定義するだけ"):
    per_customer = orders.filter(orders["status"] == "completed").groupBy("customer_id").agg(F.count("*").alias("n_orders"))

In [ ]:
with observe(spark, "action を呼ぶ"):
    print(f"{per_customer.count():,} 顧客")

✏️ **観察: 定義にかかった時間と、count() にかかった時間は？ Spark UI の Jobs タブにジョブが現れたのは、どのセルを実行したとき？**

[定義するだけ] 0.24 秒

814,438 顧客
[action を呼ぶ] 2.64 秒 / ジョブ 1 / ステージ 3
  stage  tasks          入力行    shuffle書き    shuffle読み  内容
      4      8      5,000,000        14.1 MB              -  count at NativeMethodAccessorImpl.java:0
      5    200              0        11.5 KB        14.1 MB  count at NativeMethodAccessorImpl.java:0
      6      1              0              -        11.5 KB  count at NativeMethodAccessorImpl.java:0
  ファイル読み込み: 16 ファイル / 85.0 MiB / 5,000,000 行

想定よりだいぶ早かった。
per_customer を定義したときはJobsに出てこなかった。
代わりにobserveのセルは出てきた。実際にcount()をとって計算をしたタイミングで出てきた。

---

## 実験2: explain で計画を読む

`explain(mode="extended")` は、書いたコードが実行されるまでの4段階を表示する。

| 段階 | 中身 |
|---|---|
| Parsed Logical Plan | 書いたとおりの処理の並び |
| Analyzed Logical Plan | 列名や型を解決したもの |
| Optimized Logical Plan | Spark（Catalyst）が書き換えた後の処理の並び |
| Physical Plan | 実際にどう実行するか（どの結合方式か、どこで shuffle するか） |

計画は木になっていて、**下から上に**読む。いちばん下の `FileScan` がファイルの読み込みで、データは上に向かって流れる。

In [ ]:
per_customer.explain(mode="extended")

✏️ **観察: Parsed と Optimized を見比べる。Optimized にだけ現れた演算子は何？ Physical Plan の FileScan の ReadSchema には、orders の5列のうちどの列が載っている？**

== Parsed Logical Plan ==
'Aggregate ['customer_id], ['customer_id, 'count(*) AS n_orders#38]
+- Filter (status#15 = completed)
   +- Relation [order_id#12,customer_id#13,ordered_at#14,status#15,coupon_code#16] parquet

== Analyzed Logical Plan ==
customer_id: int, n_orders: bigint
Aggregate [customer_id#13], [customer_id#13, count(1) AS n_orders#38L]
+- Filter (status#15 = completed)
   +- Relation [order_id#12,customer_id#13,ordered_at#14,status#15,coupon_code#16] parquet

== Optimized Logical Plan ==
Aggregate [customer_id#13], [customer_id#13, count(1) AS n_orders#38L]
+- Project [customer_id#13]
   +- Filter (isnotnull(status#15) AND (status#15 = completed))
      +- Relation [order_id#12,customer_id#13,ordered_at#14,status#15,coupon_code#16] parquet

== Physical Plan ==
*(2) HashAggregate(keys=[customer_id#13], functions=[count(1)], output=[customer_id#13, n_orders#38L])
+- Exchange hashpartitioning(customer_id#13, 200), ENSURE_REQUIREMENTS, [plan_id=168]
   +- *(1) HashAggregate(keys=[customer_id#13], functions=[partial_count(1)], output=[customer_id#13, count#52L])
      +- *(1) Project [customer_id#13]
         +- *(1) Filter (isnotnull(status#15) AND (status#15 = completed))
            +- *(1) ColumnarToRow
               +- FileScan parquet [customer_id#13,status#15] Batched: true, DataFilters: [isnotnull(status#15), (status#15 = completed)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/data/large/orders], PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,completed)], ReadSchema: struct<customer_id:int,status:string>


「Project」が生えた。これは単に読み込み実行のこと？
「Filter」には「isnotnull(status#15)」と、「(status#15 = completed)」が、
「Aggregate」には「customer_id#13」が新しく追加されている。
Physical Planでは customer_id, status の2つがReadSchemaになっている。フィルター条件とGROUP BY条件の2項目のみに読み込みを絞っている(Column pruning)。

In [ ]:
# TODO 2-1: 「completed な注文に、顧客の segment を付けたもの」を2通りの順番で書く
#   a: orders を先に filter してから、customers と customer_id で join する
#   b: orders と customers を join してから、filter する
#   どちらも最後に .select("order_id", "segment") する
a = orders.filter(orders["status"] == "completed").join(customers, "customer_id", "inner")
b = orders.join(customers, "customer_id", "inner").filter(orders["status"] == "completed")

In [ ]:
same_optimized_plan(a, b)
b.explain()

✏️ **観察: b では join の後に filter を書いた。Physical Plan では、Filter は join の前と後のどちらにある？ 書く順番は性能に影響した？**

✅ 最適化後の論理計画は同じ
== Physical Plan ==
*(5) Project [customer_id#13, order_id#12, ordered_at#14, status#15, coupon_code#16, name#18, prefecture#19, segment#20, registered_on#21, referrer_id#22, email#23]
+- *(5) SortMergeJoin [customer_id#13], [customer_id#17], Inner
   :- *(2) Sort [customer_id#13 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#13, 200), ENSURE_REQUIREMENTS, [plan_id=230]
   :     +- *(1) Filter ((isnotnull(status#15) AND (status#15 = completed)) AND isnotnull(customer_id#13))
   :        +- *(1) ColumnarToRow
   :           +- FileScan parquet [order_id#12,customer_id#13,ordered_at#14,status#15,coupon_code#16] Batched: true, DataFilters: [isnotnull(status#15), (status#15 = completed), isnotnull(customer_id#13)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/data/large/orders], PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,completed), IsNotNull(customer_id)], ReadSchema: struct<order_id:int,customer_id:int,ordered_at:timestamp,status:string,coupon_code:string>
   +- *(4) Sort [customer_id#17 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#17, 200), ENSURE_REQUIREMENTS, [plan_id=239]
         +- *(3) Filter isnotnull(customer_id#17)
            +- *(3) ColumnarToRow
               +- FileScan parquet [customer_id#17,name#18,prefecture#19,segment#20,registered_on#21,referrer_id#22,email#23] Batched: true, DataFilters: [isnotnull(customer_id#17)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/data/large/customers], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<customer_id:int,name:string,prefecture:string,segment:string,registered_on:date,referrer_i...

JOIN は Filter の後になっている。書く順番に影響してなさそう

---

## 実験3: 列の刈り込みとプッシュダウン

Physical Plan の `FileScan parquet` の行には、ファイルの読み方が書いてある。

| 項目 | 意味 |
|---|---|
| `ReadSchema` | 実際に読む列。Parquet は列ごとに保存されているので、使わない列は読まずに済む（列の刈り込み） |
| `PushedFilters` | Parquet の読み取り側に渡した条件。ファイル内の統計（最小値・最大値）を見て、条件に合う行がないブロックを読み飛ばせる（述語プッシュダウン） |
| `PartitionFilters` | ディレクトリ名（`order_month=2026-03` など）で絞る条件。該当しないディレクトリは開きもしない（パーティションの刈り込み） |

In [ ]:
# TODO 3-1: orders から、status が 'completed' の行の order_id と customer_id だけを取り出す
slim = orders.filter(orders["status"] == "completed").select("order_id", "customer_id")

In [ ]:
slim.explain()
check_plan(slim, has=["EqualTo(status,completed)"], lacks=["ordered_at", "coupon_code"])

次はパーティションの刈り込み。`data/large/orders_by_month` は、同じ注文データを `order_month=2025-01/` のような月ごとのディレクトリに分けて保存してある。

In [ ]:
# TODO 3-2: orders_by_month から 2026年3月の注文だけを取り出す。order_month 列（"2026-03" という文字列）で絞る
by_month = large(spark, "orders_by_month")
march = by_month.filter(F.col("order_month") == "2026-03")

In [ ]:
check_plan(march, has=["PartitionFilters: [isnotnull(order_month"])

# 比較用: 同じ「2026年3月」を、ディレクトリの列ではなく ordered_at の範囲で絞ったもの
march_by_time = by_month.filter((F.col("ordered_at") >= "2026-03-01") & (F.col("ordered_at") < "2026-04-01"))
march_by_time.explain(mode="formatted")

with observe(spark, "order_month で絞る"):
    materialize(march)
with observe(spark, "ordered_at で絞る"):
    materialize(march_by_time)

✏️ **観察: 2つの「ファイル読み込み」の行を比べる。開いたファイル数と読んだ行数はそれぞれいくつ？ ordered_at で絞ったほうの計画では、PushedFilters に条件が載っている。それなのに読んだ行数は減った？**

order_month: 1ファイル、 231,084件
orderd_at: 22ファイル、 5,000,0000件

行数は減っていない。これは全ファイルに対してorderd_atの値の範囲が分散していて全行スキャンをする必要があったから？

`PushedFilters` に載るのは「Spark が Parquet の読み取り側に条件を渡した」というだけで、実際に読み飛ばせたかどうかは別の話になる。読み飛ばしにはファイル内の統計（列の最小値・最大値）が要るが、Spark が日時を Parquet に書くときの既定の形式（INT96）には、その統計が付かない。

同じデータを、統計が付く形式（TIMESTAMP_MICROS）で書き直して比べてみる。

In [ ]:
micros_path = str(DATA / "tmp" / "orders_by_month_micros")
spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
by_month.repartition("order_month").write.mode("overwrite").partitionBy("order_month").parquet(micros_path)
spark.conf.set("spark.sql.parquet.outputTimestampType", "INT96")  # 既定値に戻す

march_micros = (
    spark.read.parquet(micros_path)
    .filter((F.col("ordered_at") >= "2026-03-01") & (F.col("ordered_at") < "2026-04-01"))
)
with observe(spark, "ordered_at で絞る（TIMESTAMP_MICROS）"):
    materialize(march_micros)

✏️ **観察: 3つの方法（order_month で絞る / ordered_at・INT96 / ordered_at・TIMESTAMP_MICROS）で、開いたファイル数と読んだ行数はどう違った？**

最後の方法は、ファイル読み込みは22・読み込み行は231,084件

order_monthはディレクトリでのパーティションなのでファイル読み込みが1件(列のプルーニング)
タイムスタンプのordered_atはファイル読み込みは走ったものの、統計データから中身の行読み込みはスキップされた？(プッシュダウン？)

ここから言えること:

- **ディレクトリによる刈り込み**は、該当しないファイルを開きもしない。いちばん強い
- **統計による読み飛ばし**は、ファイルは開くが、中身のブロックを読まずに済ませる。型やファイルの書き方によって、効くときと効かないときがある
- 計画に書いてあることと、実際に起きたことは違う場合がある。読んだ行数を見て確かめる

「よく絞る列でディレクトリを分ける」という設計はここから来ている。ただし分けすぎると小さいファイルが大量にできる（`02_partitions` で扱う）。Delta Lake の OPTIMIZE や Liquid Clustering は、統計による読み飛ばしが効きやすいようにデータを並べ直す仕組みだ（`databricks/` で扱う）。

---

## 実験4: Python の UDF を挟むと最適化が効かない

Spark が最適化できるのは、中身が分かっている処理だけ。Python の関数（UDF）は Spark から見るとブラックボックスなので、条件を Parquet に渡せないし、全行を Python のプロセスに送って1行ずつ評価することになる。

In [ ]:
is_completed = F.udf(lambda s: s == "completed", BooleanType())
slim_udf = orders.filter(is_completed("status")).select("order_id", "customer_id")

slim_udf.explain()
with observe(spark, "組み込みの比較（slim）"):
    materialize(slim)
with observe(spark, "Python UDF（slim_udf）"):
    materialize(slim_udf)

✏️ **観察: slim_udf の計画で、PushedFilters はどうなった？ slim にはなかった演算子は？ 実行時間は何倍違う？**

PushedFilterが空になった。
時間も 0.39s から 2.37s と増加。

業務で動かしている Spark のジョブ（AWS Glue など）に UDF があれば、組み込み関数（`pyspark.sql.functions`）に置き換えられないかが最初の改善候補になる。

---

## 実験5: action を呼ぶたびに最初から計算される

In [ ]:
# TODO 5-1: per_customer.count() を2回実行する。それぞれ observe で囲む
with observe(spark, "1回目"):
    per_customer.count()
with observe(spark, "2回目"):
    per_customer.count()

✏️ **観察: 2回目もファイルを読んでいる？（「ファイル読み込み」の行を見る）**

全ファイルの全行数を読んでいる。
処理時間は短くなっているが。

In [ ]:
# TODO 5-2: per_customer.cache() を呼んでから、もう一度 count() を2回実行する
per_customer.cache()
with observe(spark, "cache 後 1回目"):
    per_customer.count()
with observe(spark, "cache 後 2回目"):
    per_customer.count()

In [ ]:
per_customer.filter("n_orders >= 3").explain()

✏️ **観察: cache 後の1回目と2回目で、読んだファイル数と時間はどう変わった？ キャッシュが作られたのは cache() を呼んだとき？ それとも？ 上の計画で FileScan の代わりに現れた演算子は？ Spark UI の Storage タブには何が出ている？**

読んだファイル数0、時間も10分の1になった。
キャッシュが作られるのは1回目を実行した時になりそう？
ステージのスキップが1になっている。最初の入力行5,000,000がなくなっている
5つくらいレコードができている

In [ ]:
per_customer.unpersist()
shutil.rmtree(DATA / "tmp", ignore_errors=True)  # 実験3で書いた一時データを消す

`cache()` も transformation と同じで、呼んだ時点では何も起きない。次の action で計算した結果がメモリに残り、その次から使われる。同じ DataFrame を何度も使うときだけ意味があり、1回しか使わないなら、メモリを使うぶん損になる。

---

## まとめ

✏️ **このノートブックで分かったことを、自分の言葉で3行にまとめる**

- transformはあくまで計画、actionで実際の計算を実行する(遅延評価)
- 実行計画では大きく4つのプランがあり、その中でプルーニングやプッシュダウンなど不要な箇所をスキップし処理を高速化させる計画を確認できる
- UDFでは一部最適化が効かないものもあったり、同じtransformの使い回しはcache()が有効

### 説明できるか確かめる

答えを見ずに、声に出して説明してみる。詰まったところが、まだ分かっていないところ。

1. transformation と action は何が違う？ Spark UI のどこを見れば区別できる？
2. `df.filter(...).select(...)` と `df.select(...).filter(...)` で性能が変わらないのはなぜ？ 変わってしまうのはどんなとき？
3. `PushedFilters` と `PartitionFilters` は何が違う？
4. `cache()` が逆効果になるのはどんなとき？

In [ ]:
spark.stop()